# run

Builds the gold layer on ONE engine with dbt. `run_pipeline` calls it
twice, once per engine, after `ingest` and before `parity`:

```
ingest  ->  run (dwh)    ->  parity
        ->  run (spark)  ->
```

Five moves, a cell each: read the settings, copy the dbt project, install the adapter,
connect, build. dbt is not preinstalled in a Fabric notebook; the install cell
pip-installs it from the project's `requirements/`. Nothing here calls a script: the only
things run are `pip` and `dbt`.

Settings are in the `deploy_config` Variable Library: `engines` (`all`, `dwh` or `spark`)
and `process_limit`. The dbt project is the folder `ingest` left in
`dbt_landing/Files/project`.

**Runs in Fabric or on your laptop.** Open it in VS Code on a local Python kernel, `az login`,
and put the workspace to build in in `.env` as `FABRIC_WORKSPACE_ID`. Outside Fabric there
is no `notebookutils`, so the settings are the library's file in this repo with the `dev`
value set on top (a laptop always builds DEV), the dbt project is this repo as it is (your edits, uncommitted too), each engine gets its own venv
(`.venv-dwh`, `.venv-spark`), and every token comes from the Azure CLI.

In [ ]:
# Set by run_pipeline. Run by hand, this builds the Warehouse.
engine = "dwh"      # dwh | spark

In [ ]:
# 1. SETTINGS: the deploy_config Variable Library, as the env vars the dbt profile and the
# models read.
#
# In Fabric, notebookutils gives the library and the workspace. Anywhere else it does not
# exist: the library is read from its file in this repo, and the workspace is
# FABRIC_WORKSPACE_ID, which VS Code sets from .env.
import json
import os
import shlex
import shutil
import subprocess
import sys
import types
import urllib.request
from pathlib import Path

try:
    import notebookutils
except ImportError:
    notebookutils = None

ENGINES = ("dwh", "spark")
if engine not in ENGINES:
    raise ValueError(f"engine={engine!r}: expected {' | '.join(ENGINES)}")

if notebookutils:
    vl = notebookutils.variableLibrary.getLibrary("deploy_config")
    os.environ["FABRIC_WORKSPACE_ID"] = notebookutils.runtime.context["currentWorkspaceId"]
else:
    # The repo this notebook is in: the folder above it that holds dbt_project.yml.
    here = next(p for p in (Path.cwd(), *Path.cwd().parents) if Path(p, "dbt_project.yml").exists())
    # The library's values with the `dev` value set on top, as Fabric has them in DEV: a
    # laptop always builds DEV.
    library = Path(here, "fabric_items", "deploy_config.VariableLibrary")
    values = {v["name"]: v["value"] for v in json.loads(
        Path(library, "variables.json").read_text(encoding="utf-8"))["variables"]}
    values.update({o["name"]: o["value"] for o in json.loads(
        Path(library, "valueSets", "dev.json").read_text(encoding="utf-8"))["variableOverrides"]})
    vl = types.SimpleNamespace(**values)
    if not os.environ.get("FABRIC_WORKSPACE_ID"):
        raise RuntimeError("set FABRIC_WORKSPACE_ID in .env to the workspace to build in")
os.environ["process_limit"] = vl.process_limit
os.environ["DBT_SCHEMA"] = vl.dbt_schema
os.environ["SPARK_RESOURCE_PROFILE"] = vl.spark_resource_profile
# dbt-fabricspark 1.13.x runs OPTIMIZE after every Delta build, which rewrites the layout
# the run just produced.
os.environ["DBT_FABRICSPARK_SKIP_OPTIMIZE"] = "true"
# So dbt's log reaches the cells below line by line, not in blocks.
os.environ["PYTHONUNBUFFERED"] = "1"

if vl.engines not in ("all", engine):
    skipped = f"{engine} skipped: engines={vl.engines}"
    if notebookutils:
        notebookutils.notebook.exit(skipped)
    raise SystemExit(skipped)

In [ ]:
# 2. THE PROJECT: the dbt models, macros and tests.
#
# In Fabric, the folder `ingest` left in the landing lakehouse, this notebook's default one:
# downloaded once from GitHub, or uploaded by a deploy from CI. Copied to local disk, where
# dbt writes its target/ and logs/. On a laptop, this repo as it is, so what builds is what
# you just edited.
PROJECT = "/lakehouse/default/Files/project"
if notebookutils:
    if not os.path.exists(f"{PROJECT}/COMMIT"):
        raise RuntimeError(f"no dbt project in {PROJECT}: run `ingest`, or run_pipeline, first")
    # The ~135 GiB work disk, not the 19 GiB /tmp overlay.
    work = "/home/trusted-service-user/work" if os.path.isdir("/home/trusted-service-user/work") else "/tmp"
    repo = os.path.join(work, "fabric-medallion-dbt")
    shutil.rmtree(repo, ignore_errors=True)
    shutil.copytree(PROJECT, repo)
    commit = Path(repo, "COMMIT").read_text().strip()
else:
    repo = str(here)
    commit = "the working tree"
print(f"{repo} @ {commit}")

In [ ]:
# 3. INSTALL dbt and this engine's adapter, from the list CI tests the project with:
#
#   dwh     requirements/dwh.txt     dbt-fabric
#   spark   requirements/spark.txt   dbt-fabricspark
#
# ONE ADAPTER PER ENVIRONMENT. The two shadow each other under dbt.adapters. In Fabric,
# run_pipeline runs this notebook once per engine, each in its own session, and the adapter
# goes into the session. On a laptop the kernel lives on and may build the other engine
# next, so each engine gets its own venv in the repo (.venv-dwh, .venv-spark, gitignored)
# and the kernel's own environment is left alone. dbt then runs as a subprocess: this
# kernel never imports it, so there is nothing to restart after the install.
def sh(*cmd, check=True):
    """One command in the project folder, its log printed here line by line."""
    p = subprocess.Popen(list(cmd), cwd=repo, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() and check:
        # Raise so a scheduled run goes red instead of silently reporting success.
        raise RuntimeError(f"{shlex.join(cmd)} failed (exit {p.returncode}); see the log above")
    return p.returncode


if notebookutils:
    python = sys.executable
else:
    venv = Path(repo, f".venv-{engine}")
    if not venv.exists():
        sh(sys.executable, "-m", "venv", str(venv))
    python = str(venv / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))
sh(python, "-m", "pip", "install", "-q", "-r", f"requirements/{engine}.txt")

In [ ]:
# 4. CONNECT: tell the dbt profile (profiles.yml) where this engine's warehouse or lakehouse
# is. The install created the items; their ids are looked up by name, with the Fabric REST
# API.
workspace = os.environ["FABRIC_WORKSPACE_ID"]
onelake = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com"


def token():
    """A Fabric REST token for whoever runs the notebook: notebookutils in Fabric, the Azure
    CLI's `az login` anywhere else."""
    if notebookutils:
        return notebookutils.credentials.getToken("pbi")
    return subprocess.run(
        [shutil.which("az") or "az", "account", "get-access-token",
         "--resource", "https://api.fabric.microsoft.com", "--query", "accessToken", "-o", "tsv"],
        check=True, capture_output=True, text=True).stdout.strip()


def fabric(method, path, body=None):
    """One call to the Fabric REST API for this workspace, as whoever runs the notebook."""
    request = urllib.request.Request(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace}{path}", method=method,
        data=json.dumps(body).encode() if body else None,
        headers={"Authorization": f"Bearer {token()}", "Content-Type": "application/json"})
    with urllib.request.urlopen(request) as r:
        return json.loads(r.read() or b"{}")


def item(kind, name):
    found = [i for i in fabric("GET", f"/{kind}")["value"] if i["displayName"] == name]
    if not found:
        raise RuntimeError(f"{name} is not in this workspace; install the project first")
    return found[0]["id"]


landing, lakehouse = item("lakehouses", "dbt_landing"), item("lakehouses", "dbt")
os.environ["FABRIC_WORKSPACE_NAME"] = fabric("GET", "")["displayName"]

if engine == "dwh":
    warehouse = fabric("GET", f"/warehouses/{item('warehouses', 'dbt_dwh')}")
    os.environ["FABRIC_DWH_NAME"] = warehouse["displayName"]
    os.environ["FABRIC_DWH_SERVER"] = warehouse["properties"]["connectionString"]
    # A Warehouse has no Files section, so it reads the landed CSVs through a shortcut: the
    # `dbt` lakehouse's Files/landing points at dbt_landing/Files. Written when it is missing
    # or points somewhere else, which is what a deleted and recreated dbt_landing leaves.
    target = {"workspaceId": workspace, "itemId": landing, "path": "Files"}
    shortcuts = fabric("GET", f"/items/{lakehouse}/shortcuts").get("value", [])
    if not any(s["name"] == "landing" and s["target"].get("oneLake") == target for s in shortcuts):
        fabric("POST", f"/items/{lakehouse}/shortcuts?shortcutConflictPolicy=CreateOrOverwrite",
               {"path": "Files", "name": "landing", "target": {"oneLake": target}})
    os.environ["FILES_PATH"] = f"{onelake}/{lakehouse}/Files/landing"
else:
    os.environ["FABRIC_LAKEHOUSE_ID"] = lakehouse
    os.environ["FABRIC_LAKEHOUSE_NAME"] = "dbt"
    # Spark reads the landing zone directly: no shortcut, no second copy.
    os.environ["FILES_PATH"] = f"{onelake}/{landing}/Files"

# In Fabric each adapter asks notebookutils for the token, and each has its own name for
# that. Anywhere else both take it from the Azure CLI.
if notebookutils:
    os.environ["FABRIC_AUTH"] = {"dwh": "notebookutils", "spark": "fabric_notebook"}[engine]
else:
    os.environ["FABRIC_AUTH"] = "CLI"
for k in ("FABRIC_DWH_SERVER", "FABRIC_DWH_NAME", "FABRIC_LAKEHOUSE_NAME", "FILES_PATH"):
    if k in os.environ:
        print(f"{k}={os.environ[k]}")

In [ ]:
# 5. BUILD: every model, then every test.
def dbt(*args):
    # `python -m`, not the `dbt` executable: pip may put that outside the notebook's PATH.
    return [python, "-m", "dbt.cli.main", *args, "--target", engine, "--profiles-dir", "."]


# No --select: what is built is decided by +enabled in dbt_project.yml, on the target.
if sh(*dbt("build"), check=False):
    sh(*dbt("retry"))